# Week 12 Mini Graded Project — E-Commerce Order, Refund & Exception Handling with CrewAI

This notebook is a **fully corrected runnable version** of the Week 12 graded mini project using **CrewAI** with the **Together provider**.

## Fixes in this version
- Uses Together provider in CrewAI-compatible format
- Avoids passing `ChatTogether(...)` into agents
- Uses separate code blocks for each agent
- Uses `verbose=True` instead of `verbose=2`
- Fixes the Jupyter/Colab event-loop error by using **async Crew execution**


## Step 1 — Install Required Libraries

If you previously installed conflicting packages in the same runtime, restart the runtime first and then run this notebook from the top.


In [ ]:
!pip install -q crewai litellm python-dotenv nest_asyncio

## Step 2 — Load `.env` and Configure Together Provider

Expected `.env` example:
```env
LLM_PROVIDER=together
TOGETHER_API_KEY=your_key_here
LLM_MODEL_DEFAULT=Qwen/Qwen2.5-7B-Instruct-Turbo
```


In [ ]:
import os
import json
import asyncio
import warnings
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio

warnings.filterwarnings('ignore')
nest_asyncio.apply()

def _load_env():
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        env_path = parent / '.env'
        if env_path.exists():
            load_dotenv(env_path)
            return env_path
    load_dotenv()
    return None

env_path = _load_env()
print(f'.env loaded from: {env_path}')

PROVIDER = os.getenv('LLM_PROVIDER', 'together').lower()
print('Provider:', PROVIDER)

if PROVIDER != 'together':
    raise ValueError('This notebook is configured for Together provider only.')

together_api_key = os.getenv('TOGETHER_API_KEY')
if not together_api_key:
    raise ValueError('TOGETHER_API_KEY is missing from your .env file')

model_name = os.getenv('LLM_MODEL_DEFAULT', 'Qwen/Qwen2.5-7B-Instruct-Turbo')

os.environ['TOGETHERAI_API_KEY'] = together_api_key
crewai_llm = f'together_ai/{model_name}'

print('Model name:', model_name)
print('CrewAI LLM reference:', crewai_llm)

In [ ]:
from crewai import Agent, Task, Crew

## Step 3 — Define Static E-Commerce Policy


In [ ]:
ECOMMERCE_POLICY = '''
E-Commerce Policy Rules:

1. Standard return window:
   - Items can be returned within 7 days of delivery.

2. Damaged item policy:
   - If the item arrives damaged, the customer is eligible for replacement or refund.

3. Wrong item policy:
   - If the wrong item is delivered, the customer is eligible for replacement or refund.

4. Delayed shipment policy:
   - If an order is not delivered and is delayed by more than 5 days beyond expected delivery,
     the customer may cancel for a refund.
   - If still in transit and delay is small, advise waiting.

5. Non-returnable items:
   - Grocery items
   - Personal care products
   - Custom-made products

6. Opened electronics:
   - If damaged or wrong item, replacement is allowed.
   - Refund may require manual review.

7. High-value review threshold:
   - Orders above $500 must be manually reviewed before final refund approval.

8. Delivered but not received:
   - Must be escalated for manual investigation.

9. Repeat refund behavior:
   - More than 2 refund requests in the last 30 days should be escalated.

10. Missing information:
   - If the complaint lacks enough detail, ask for clarification before resolution.

11. Resolution types allowed:
   - Refund
   - Replacement
   - Wait / monitor shipment
   - Reject request
   - Ask for more information
   - Escalate for manual review
'''

## Step 4 — Create Mock Order Database


In [ ]:
mock_orders = {
    '1001': {'customer_name': 'Aarav', 'item': 'Wireless Earbuds', 'category': 'electronics', 'price': 80, 'status': 'in_transit', 'expected_delivery_days_late': 6, 'delivered_days_ago': None, 'opened': False, 'prior_refund_requests_30d': 0},
    '1002': {'customer_name': 'Meera', 'item': 'Blender', 'category': 'home_appliance', 'price': 120, 'status': 'delivered', 'expected_delivery_days_late': 0, 'delivered_days_ago': 0, 'opened': True, 'prior_refund_requests_30d': 0},
    '1003': {'customer_name': 'Rahul', 'item': 'Running Shoes', 'category': 'fashion', 'price': 65, 'status': 'delivered', 'expected_delivery_days_late': 0, 'delivered_days_ago': 0, 'opened': False, 'prior_refund_requests_30d': 0},
    '1004': {'customer_name': 'Sana', 'item': 'Office Chair', 'category': 'furniture', 'price': 150, 'status': 'delivered', 'expected_delivery_days_late': 0, 'delivered_days_ago': 10, 'opened': False, 'prior_refund_requests_30d': 0},
    '1005': {'customer_name': 'Kiran', 'item': 'Skin Care Cream', 'category': 'personal_care', 'price': 25, 'status': 'delivered', 'expected_delivery_days_late': 0, 'delivered_days_ago': 2, 'opened': True, 'prior_refund_requests_30d': 0},
    '1006': {'customer_name': 'Nisha', 'item': 'Laptop', 'category': 'electronics', 'price': 1200, 'status': 'delivered', 'expected_delivery_days_late': 0, 'delivered_days_ago': 1, 'opened': True, 'prior_refund_requests_30d': 1},
    '1007': {'customer_name': 'Vikram', 'item': 'Smartphone', 'category': 'electronics', 'price': 700, 'status': 'delivered', 'expected_delivery_days_late': 0, 'delivered_days_ago': 1, 'opened': False, 'prior_refund_requests_30d': 0}
}

def get_order_context(order_id):
    if order_id in mock_orders:
        return json.dumps(mock_orders[order_id], indent=2)
    return 'Order ID not found in mock database.'

print('Mock orders loaded:', len(mock_orders))

# Separate Agent Blocks


In [ ]:
issue_agent = Agent(
    role='Order Issue Identification Agent',
    goal='Identify the exact customer issue, extract key facts, and structure the case for downstream agents.',
    backstory='You are an e-commerce support triage specialist who identifies delay, damage, wrong item, refund, and ambiguity cases.',
    llm=crewai_llm,
    verbose=True
)
print('Issue agent created successfully')

In [ ]:
policy_agent = Agent(
    role='Policy Interpretation Agent',
    goal='Apply return, refund, delay, and exception policies consistently using the structured issue summary and order details.',
    backstory='You are an expert in e-commerce policies and interpret static business rules consistently.',
    llm=crewai_llm,
    verbose=True
)
print('Policy agent created successfully')

In [ ]:
resolution_agent = Agent(
    role='Resolution Recommendation Agent',
    goal='Recommend the most appropriate customer resolution using the issue summary and policy decision.',
    backstory='You translate policy outcomes into actionable customer support responses.',
    llm=crewai_llm,
    verbose=True
)
print('Resolution agent created successfully')

In [ ]:
escalation_agent = Agent(
    role='Escalation Agent',
    goal='Decide whether the case requires manual review based on risk, urgency, cost, uncertainty, or policy exceptions.',
    backstory='You protect operations by escalating risky, expensive, uncertain, and policy-sensitive cases.',
    llm=crewai_llm,
    verbose=True
)
print('Escalation agent created successfully')

## Step 5 — Define Tasks


In [ ]:
issue_task = Task(
    description='''
Customer query: {customer_query}
Order ID: {order_id}
Order details: {order_context}

Identify issue type, extract key facts, customer intent, urgency, and missing information.
''',
    expected_output='A structured issue identification summary.',
    agent=issue_agent
)

policy_task = Task(
    description='''
Customer query: {customer_query}
Order details: {order_context}
Policy: {policy_text}

Apply the policy and determine eligibility, applicable rules, allowed resolutions, exceptions, and uncertainty.
''',
    expected_output='A policy interpretation summary with eligibility and applicable rules.',
    agent=policy_agent
)

resolution_task = Task(
    description='''
Customer query: {customer_query}
Order details: {order_context}
Policy: {policy_text}

Recommend the best resolution and draft a customer-facing response.
''',
    expected_output='A clear resolution recommendation and customer-facing message draft.',
    agent=resolution_agent
)

escalation_task = Task(
    description='''
Customer query: {customer_query}
Order details: {order_context}
Policy: {policy_text}

Decide whether escalation is required based on high value, uncertainty, missing evidence, delivered-but-not-received, repeated refunds, or electronics refund review.
''',
    expected_output='A final escalation decision with justification.',
    agent=escalation_agent
)

print('Tasks created successfully')

In [ ]:
support_crew = Crew(
    agents=[issue_agent, policy_agent, resolution_agent, escalation_agent],
    tasks=[issue_task, policy_task, resolution_task, escalation_task],
    verbose=True
)

print('Crew pipeline ready')

## Step 6 — Async Runner Function

### Why async?
In Jupyter/Colab, a running event loop may already exist.

So instead of `support_crew.kickoff(...)`, we use:
- `await support_crew.kickoff_async(...)`

This avoids the runtime error you encountered.


In [ ]:
async def run_case_async(order_id, customer_query):
    order_context = get_order_context(order_id)
    inputs = {
        'order_id': order_id,
        'customer_query': customer_query,
        'order_context': order_context,
        'policy_text': ECOMMERCE_POLICY
    }

    result = await support_crew.kickoff_async(inputs=inputs)

    print('\n' + '=' * 90)
    print(f'ORDER ID: {order_id}')
    print(f'QUERY: {customer_query}')
    print('-' * 90)
    print('FINAL OUTPUT:')
    print(result)
    print('=' * 90 + '\n')
    return result

## Step 7 — Representative Test Cases


In [ ]:
test_cases = [
    ('1001', 'My order #1001 was supposed to arrive 6 days ago and still has not been delivered. I want to cancel and get a refund.'),
    ('1002', 'I received order #1002 today, but the blender jar is cracked and unusable. I want a replacement.'),
    ('1003', 'Order #1003 delivered today, but I ordered black shoes and got white shoes. Please fix this.'),
    ('1004', 'I got order #1004 ten days ago and now I want to return it because I changed my mind.'),
    ('1005', 'I want to return the skin-care product from order #1005 because I do not like it.'),
    ('1006', 'My order #1006 laptop arrived damaged and the screen is broken. I need a refund immediately.'),
    ('1007', 'Tracking says my order #1007 was delivered, but I never got it.'),
    ('9999', 'My order is wrong and I want my money back.')
]

print('Total test cases:', len(test_cases))

## Step 8 — Run One Sample Case

Use `await` in notebook cells.


In [ ]:
await run_case_async(*test_cases[0])

## Step 9 — Run All Test Cases


In [ ]:
all_results = []
for case in test_cases:
    result = await run_case_async(*case)
    all_results.append(result)

# Architecture Explanation

## 1. Overview
This project implements a goal-oriented multi-agent workflow for e-commerce customer support using CrewAI. The system handles customer issues related to delayed deliveries, damaged items, wrong items, refund eligibility, and policy exceptions.

## 2. Agent Roles
- Order Issue Identification Agent
- Policy Interpretation Agent
- Resolution Recommendation Agent
- Escalation Agent

## 3. Task Flow / Handoffs
The workflow is sequential: issue identification → policy interpretation → resolution recommendation → escalation decision.

## 4. Escalation Logic
Cases are escalated for high value, missing delivery conflicts, uncertainty, repeated refunds, or electronics refund review.

## 5. Design Rationale
The design uses specialized agents, clear handoffs, static policy rules, and explicit escalation thresholds.
